# 🚀 Phase 3: Prescriptive Analytics & Optimization Engine
**Agritech Decision Intelligence: Lettuce Yield Optimization**

> **🇬🇧 EN:** Welcome to the final tier of Decision Intelligence: Prescriptive Analytics. We transition from predicting the future to actively shaping it. This engine ingests our serialized Random Forest model and real-time environmental data (non-controllable variables) to simulate thousands of biochemical combinations. It then outputs the exact operational commands (controllable levers: pH and EC) required by the IoT actuators to maximize crop yield.
>
> **🇲🇽 ES:** Bienvenido al nivel final de la Inteligencia de Decisiones: Analítica Prescriptiva. Pasamos de predecir el futuro a darle forma activamente. Este motor ingiere nuestro modelo Random Forest serializado y datos ambientales en tiempo real (variables incontrolables) para simular miles de combinaciones bioquímicas. Luego, emite los comandos operativos exactos (palancas controlables: pH y EC) requeridos por los actuadores IoT para maximizar el rendimiento del cultivo.

In [2]:
# 1. Imports & MLOps Logging Setup
import pandas as pd
import numpy as np
import joblib
import logging
import os
import json

# Configure MLOps Logger for the Prescriptive Engine
log_dir = '../logs/'
os.makedirs(log_dir, exist_ok=True)
logging.basicConfig(
    filename=os.path.join(log_dir, 'prescriptive_engine.log'),
    level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s'
)
logging.info("--- Phase 3: Prescriptive Optimization Engine Initialized ---")
print("✅ Libraries loaded. MLOps Logging active (/logs/prescriptive_engine.log).")

# 2. Ingest the Serialized Predictive Artifact
model_path = '../models/rf_yield_predictor.pkl'
try:
    rf_model = joblib.load(model_path)
    logging.info(f"Predictive model loaded successfully from {model_path}")
    print(f"✅ Advanced Predictive Engine (Random Forest) loaded successfully.")
except FileNotFoundError:
    print(f"❌ ERROR: Model artifact not found at {model_path}. Please run Notebook 02 first.")

✅ Libraries loaded. MLOps Logging active (/logs/prescriptive_engine.log).
✅ Advanced Predictive Engine (Random Forest) loaded successfully.


## 🛠️ 1. Architecture of the Optimization Engine
**🇬🇧 EN:** To optimize a non-linear system, we split our features into two categories:
1. **Non-Controllable (Environmental Base):** Temperature, Humidity, Light Hours, Growth Days.
2. **Controllable (Operational Levers):** pH Level, Nutrient EC.

The algorithm will lock the environmental base and generate a multidimensional grid of the operational levers. It predicts the yield for every possible combination and extracts the `argmax` (the combination yielding the highest crop weight).
**🇲🇽 ES:** Para optimizar un sistema no lineal, dividimos nuestras características en dos categorías:
1. **No Controlables (Base Ambiental):** Temperatura, Humedad, Horas de Luz, Días de Crecimiento.
2. **Controlables (Palancas Operativas):** Nivel de pH, EC de Nutrientes.

El algoritmo bloqueará la base ambiental y generará una cuadrícula multidimensional de las palancas operativas. Predecirá el rendimiento para cada combinación posible y extraerá el `argmax` (la combinación que produce el mayor peso).

In [4]:
def prescriptive_optimization_engine(current_temp, current_hum, current_light, current_days, current_ph, current_ec):
    """
    Simulates operational adjustments to maximize yield based on current environmental conditions.
    """
    logging.info(f"Engine called with Env: Temp={current_temp}, Hum={current_hum} | State: pH={current_ph}, EC={current_ec}")
    
    # 1. Baseline Prediction (Current State)
    # Feature order MUST match Notebook 02: ['Temperature_C', 'Humidity_percent', 'pH_Level', 'Nutrient_EC_mS', 'Light_Hours', 'Growth_Days']
    baseline_features = pd.DataFrame([[current_temp, current_hum, current_ph, current_ec, current_light, current_days]], 
                                     columns=['Temperature_C', 'Humidity_percent', 'pH_Level', 'Nutrient_EC_mS', 'Light_Hours', 'Growth_Days'])
    baseline_yield = rf_model.predict(baseline_features)[0]
    
    # 2. Simulation Grid Generation (Action Space)
    # Searching pH from 4.0 to 8.0 (step 0.1), and EC from 0.5 to 2.5 (step 0.1)
    ph_space = np.arange(4.0, 8.1, 0.1)
    ec_space = np.arange(0.5, 2.6, 0.1)
    
    # Create all possible combinations
    grid_ph, grid_ec = np.meshgrid(ph_space, ec_space)
    simulation_df = pd.DataFrame({
        'Temperature_C': current_temp,
        'Humidity_percent': current_hum,
        'pH_Level': grid_ph.flatten(),
        'Nutrient_EC_mS': grid_ec.flatten(),
        'Light_Hours': current_light,
        'Growth_Days': current_days
    })
    
    # 3. Mass Prediction & Argmax Extraction
    simulation_df['Predicted_Yield'] = rf_model.predict(simulation_df[['Temperature_C', 'Humidity_percent', 'pH_Level', 'Nutrient_EC_mS', 'Light_Hours', 'Growth_Days']])
    
    # Find the row with the maximum predicted yield
    optimal_row = simulation_df.loc[simulation_df['Predicted_Yield'].idxmax()]
    
    # 4. Calculate Delta (Value Added)
    optimized_yield = optimal_row['Predicted_Yield']
    yield_gained = optimized_yield - baseline_yield
    
    # Compile Results
    recommendation = {
        "current_state": {"pH": current_ph, "EC": current_ec, "predicted_yield_g": round(baseline_yield, 2)},
        "recommended_action": {"set_pH": round(optimal_row['pH_Level'], 2), "set_EC": round(optimal_row['Nutrient_EC_mS'], 2)},
        "optimized_projection": {"predicted_yield_g": round(optimized_yield, 2), "net_gain_g": round(yield_gained, 2)}
    }
    
    logging.info(f"Optimization successful. Net Gain: +{round(yield_gained, 2)}g")
    return recommendation

print("✅ Prescriptive Optimization Engine compiled and ready for IoT edge deployment.")

✅ Prescriptive Optimization Engine compiled and ready for IoT edge deployment.


## 🧪 2. Business Value Demonstration (Scenario Simulation)
**🇬🇧 EN:** We simulate a real-world scenario where a greenhouse is currently operating at suboptimal biochemical levels. The engine will evaluate the environment and prescribe the exact mechanical adjustments to rescue the crop yield.
**🇲🇽 ES:** Simulamos un escenario del mundo real donde un invernadero está operando actualmente a niveles bioquímicos subóptimos. El motor evaluará el entorno y prescribirá los ajustes mecánicos exactos para rescatar el rendimiento del cultivo.

In [6]:
# Scenario: Greenhouse 4 is currently experiencing high pH and low nutrients.
scenario_temp = 22.5
scenario_hum = 60.0
scenario_light = 14
scenario_days = 45

suboptimal_ph = 7.5  # Too alkaline
suboptimal_ec = 0.8  # Too low in nutrients

print("📡 [IoT SENSOR FEED] Intercepting current greenhouse metrics...")
print(f"Environment: Temp {scenario_temp}°C | Humidity {scenario_hum}%\n")

# Run the Optimization Engine
decision_payload = prescriptive_optimization_engine(
    current_temp=scenario_temp, 
    current_hum=scenario_hum, 
    current_light=scenario_light, 
    current_days=scenario_days, 
    current_ph=suboptimal_ph, 
    current_ec=suboptimal_ec
)

# Format the output as a professional JSON payload (Ready for API integration)
print("🧠 [DECISION ENGINE] Executing Multidimensional Simulation...")
print(json.dumps(decision_payload, indent=4))

# Executive Summary
gain = decision_payload['optimized_projection']['net_gain_g']
pct_improvement = (gain / decision_payload['current_state']['predicted_yield_g']) * 100

print(f"\n💰 [EXECUTIVE SUMMARY]")
print(f"By executing the recommended actions, the farm will increase yield by {gain} grams per plant.")
print(f"This represents a mathematically validated production efficiency increase of {pct_improvement:.1f}%.")

📡 [IoT SENSOR FEED] Intercepting current greenhouse metrics...
Environment: Temp 22.5°C | Humidity 60.0%

🧠 [DECISION ENGINE] Executing Multidimensional Simulation...
{
    "current_state": {
        "pH": 7.5,
        "EC": 0.8,
        "predicted_yield_g": 219.95
    },
    "recommended_action": {
        "set_pH": 6.4,
        "set_EC": 1.6
    },
    "optimized_projection": {
        "predicted_yield_g": 266.13,
        "net_gain_g": 46.19
    }
}

💰 [EXECUTIVE SUMMARY]
By executing the recommended actions, the farm will increase yield by 46.19 grams per plant.
This represents a mathematically validated production efficiency increase of 21.0%.


## 📬 3. Contact
**Pablo Alberto Santana Flores**
*Data Scientist | PhDc in Marine Sciences | Chemical Engineer*

* 💼 **LinkedIn:** [linkedin.com/in/pablo-santana-mx](https://www.linkedin.com/in/pablo-santana-mx)
* 🐙 **GitHub:** [github.com/Pablo-Santana-MX](https://github.com/Pablo-Santana-MX)